# Improved COST training, snapshot prediction, architectural/optimization ablation

This notebook contains code selected from the authors’ research notebooks because its analysis is described in the accepted COST manuscript. Outputs and execution counts have been removed. It may require authorized datasets, saved model artifacts, and variables established by preceding pipeline notebooks. See `docs/EXECUTION_ORDER.md`.


**Source verification:** The model, training, snapshot ensemble and ablation definitions are also present in the uploaded `DS_RNN_v2_Improvements (2)(1).ipynb`. This notebook retains the paper-aligned pipeline from the original COST master; avoid mixing checkpoint formats between historical variants.


## Bagian 13: Configuration v2 (COST Improved)

In [ ]:
N_INPUT = N_FEATURES * 3   # value + missingness mask + delta-t

CONFIG_V2 = {
    'hidden_size'    : 256,
    'n_lstm_layers'  : 2,
    'n_heads'        : 4,
    'attention_dim'  : 128,
    'salience_hidden': 128,
    'dropout'        : 0.3,
    'sparsity_tau'   : 0.5,
    'lambda_sparse'  : 5e-4,
    'lr'             : 3e-4,
    'weight_decay'   : 1e-4,
    'batch_size'     : 64,
    'max_epochs'     : 120,
    'patience'       : 15,
    'grad_clip'      : 1.0,
    'warmup_pct'     : 0.1,
    'focal_gamma'    : 2.0,
    'focal_alpha'    : 0.75,
    'label_smoothing': 0.05,
    'stoch_depth_p'  : 0.1,
    'rdrop_alpha'    : 4.0,
    'n_snapshots'    : 5,
    'n_runs'         : 3,
    'seq_len'        : 48,
    'train_years'    : range(2008, 2016),
    'val_years'      : range(2016, 2018),
    'test_years'     : range(2018, 2020),
}

print('\u2705 Konfigurasi v2 dimuat')
print(f'   Input dim per timestep : {N_INPUT} (nilai + mask + delta_t)')
print(f'   Hidden size            : {CONFIG_V2["hidden_size"]}')

## Bagian 14: Preprocessing v2 (missingness mask + delta-t features) — reassigns `splits`/`loaders`

In [ ]:
# Reuse already-loaded raw arrays from Bagian 3 (no need to re-read cache from disk)
X_temporal_raw, X_static_raw, labels_raw, years_raw = X_temporal, X_static, labels, years

def add_missingness_features(X_temporal):
    n, seq, feats = X_temporal.shape
    mask = (~np.isnan(X_temporal)).astype(np.float32)

    delta = np.zeros_like(X_temporal, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X_temporal[i, t, k]):
                    last_obs = t
                    delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)

    values = X_temporal.copy()
    n_vital = len(VITAL_SIGNS)
    for i in range(n):
        for k in range(n_vital):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)

    X_enhanced = np.concatenate([values, mask, delta], axis=-1)
    return X_enhanced.astype(np.float32)


def preprocess_v2(X_temporal, X_static, labels, years, config):
    train_mask = np.isin(years, list(config['train_years']))
    val_mask   = np.isin(years, list(config['val_years']))
    test_mask  = np.isin(years, list(config['test_years']))

    splits_raw = {}
    for name, mask in [('train', train_mask), ('val', val_mask), ('test', test_mask)]:
        splits_raw[name] = {
            'X_temp': X_temporal[mask].copy(),
            'X_stat': X_static[mask].copy(),
            'y'     : labels[mask].copy()
        }
        print(f'   {name:5s}: {mask.sum():,} pasien | pos: {labels[mask].sum()} ({labels[mask].mean()*100:.1f}%)')

    print('\n\u23f3 MICE imputation lab values...')
    n_vital = len(VITAL_SIGNS)
    imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)

    tr = splits_raw['train']['X_temp']
    n_tr, seq, n_feat = tr.shape
    lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
    lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)
    imp.fit(lab_train)

    for name in splits_raw:
        arr = splits_raw[name]['X_temp']
        n_sp = arr.shape[0]
        lab_d = arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital)
        lab_d = np.nan_to_num(lab_d, nan=0.0, posinf=0.0, neginf=0.0)
        arr[:, :, n_vital:] = imp.transform(lab_d).reshape(n_sp, seq, n_feat - n_vital)
        splits_raw[name]['X_temp'] = arr
    print('   \u2705 MICE selesai')

    print('\n Menambahkan missingness mask & delta-t features...')
    splits = {}
    for name in splits_raw:
        enhanced = add_missingness_features(splits_raw[name]['X_temp'])
        splits[name] = {
            'X_temp': enhanced,
            'X_stat': splits_raw[name]['X_stat'].copy(),
            'y'     : splits_raw[name]['y'].copy()
        }
    print(f'   Input shape baru: {splits["train"]["X_temp"].shape}')

    tr_flat = splits['train']['X_temp'][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
    feat_mean = tr_flat.mean(axis=0)
    feat_std  = tr_flat.std(axis=0) + 1e-8
    stat_mean = splits['train']['X_stat'].mean(axis=0)
    stat_std  = splits['train']['X_stat'].std(axis=0) + 1e-8

    for name in splits:
        arr = splits[name]['X_temp'].copy()
        arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
        splits[name]['X_temp'] = arr
        splits[name]['X_stat'] = (splits[name]['X_stat'] - stat_mean) / stat_std

    norm_stats = {'feat_mean': feat_mean, 'feat_std': feat_std,
                  'stat_mean': stat_mean, 'stat_std': stat_std}
    return splits, norm_stats


print('\u2699\ufe0f Preprocessing v2...')
splits, norm_stats = preprocess_v2(X_temporal_raw, X_static_raw, labels_raw, years_raw, CONFIG_V2)

for name, data in splits.items():
    print(f'   {name}: X_temp={data["X_temp"].shape} | X_stat={data["X_stat"].shape}')


class ICUDataset(Dataset):
    def __init__(self, X_temporal, X_static, y):
        self.X_temp = torch.FloatTensor(X_temporal)
        self.X_stat = torch.FloatTensor(X_static)
        self.y      = torch.FloatTensor(y)
    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X_temp[idx], self.X_stat[idx], self.y[idx]


def make_loaders_v2(splits, batch_size):
    loaders = {}
    for name, data in splits.items():
        ds = ICUDataset(data['X_temp'], data['X_stat'], data['y'])
        if name == 'train':
            y = data['y']
            w = 1.0 / np.bincount(y.astype(int))[y.astype(int)]
            sampler = torch.utils.data.WeightedRandomSampler(w, len(y), replacement=True)
            loaders[name] = DataLoader(ds, batch_size=batch_size, sampler=sampler,
                                       num_workers=2, pin_memory=True)
        else:
            loaders[name] = DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)
    return loaders


loaders = make_loaders_v2(splits, CONFIG_V2['batch_size'])
print('\u2705 DataLoaders v2 siap')
for name, dl in loaders.items():
    print(f'   {name}: {len(dl)} batches')

## Bagian 15: Architecture — COST (Improved) / DSRNNv2

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=256, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class StochasticDepth(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x, residual):
        if not self.training or self.p == 0:
            return x + residual
        keep = torch.bernoulli(torch.tensor(1 - self.p)).item()
        return x + residual * keep


class ClinicalSalienceV2(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_features):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim * 2)
        self.norm  = nn.LayerNorm(hidden_dim)
        self.fc2   = nn.Linear(hidden_dim, n_features)
        self.tau   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x_t, h_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)
        h = self.fc1(combined)
        h, gate = h.chunk(2, dim=-1)
        h = h * torch.sigmoid(gate)
        h = self.norm(h)
        s_t = torch.sigmoid(self.fc2(h))
        m_hard = (s_t > self.tau).float()
        m_t = m_hard - s_t.detach() + s_t
        return m_t, s_t


class TemporalMultiHeadAttn(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(d_model)
        self.sdrop = StochasticDepth(p=0.1)

    def forward(self, x):
        out, weights = self.attn(x, x, x)
        x = self.sdrop(x, out)
        x = self.norm(x)
        return x, weights


class StaticTemporalFusion(nn.Module):
    def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
        super().__init__()
        self.static_proj = nn.Linear(n_static, d_temporal)
        self.cross_attn  = nn.MultiheadAttention(embed_dim=d_temporal, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_temporal)
        self.ffn  = nn.Sequential(
            nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_temporal, d_out)
        )

    def forward(self, static_feat, hidden_states):
        q = self.static_proj(static_feat).unsqueeze(1)
        ctx, attn_w = self.cross_attn(q, hidden_states, hidden_states)
        ctx = self.norm(ctx.squeeze(1))
        pooled = hidden_states.mean(dim=1)
        z = self.ffn(torch.cat([ctx, pooled], dim=-1))
        return z, attn_w.squeeze(1)


class DSRNNv2(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden,
                 dropout=0.3, stoch_depth_p=0.1, lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size   = hidden_size
        self.n_lstm_layers = n_lstm_layers
        self.lambda_sparse = lambda_sparse
        self.bidir_factor  = 2

        self.input_proj = nn.Sequential(
            nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
        self.salience = ClinicalSalienceV2(hidden_size + hidden_size, salience_hidden, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if n_lstm_layers > 1 else 0)
        bidir_dim = hidden_size * 2
        self.bidir_proj = nn.Sequential(
            nn.Linear(bidir_dim, hidden_size), nn.LayerNorm(hidden_size), nn.GELU()
        )
        self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, p in m.named_parameters():
                    if 'weight' in name: nn.init.orthogonal_(p)
                    elif 'bias' in name: nn.init.zeros_(p)

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        x_proj = self.input_proj(x_temporal)
        x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        for t in range(seq):
            x_t = x_proj[:, t, :]
            m_t, s_t = self.salience(x_t, h_prev)
            x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1
            sparsity_loss = sparsity_loss + m_t.norm(p=1)

        x_gated_seq = torch.cat(gated_seq, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out    = self.bidir_proj(lstm_out)
        H, attn_weights = self.temporal_attn(lstm_out)
        z, cross_attn_w = self.fusion(x_static, H)
        logits = self.classifier(z).squeeze(-1)

        if return_states:
            return logits, sparsity_loss, H, attn_weights
        return logits, sparsity_loss


print('\u2705 DS-RNN v2 (COST Improved) architecture defined')

## Bagian 16: Loss Functions & Training Utilities v2

In [ ]:
class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.smoothing = alpha, gamma, smoothing

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        prob  = torch.sigmoid(logits)
        p_t   = prob * targets + (1 - prob) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


def rdrop_kl_loss(logits1, logits2):
    p1 = torch.sigmoid(logits1); p2 = torch.sigmoid(logits2)
    eps = 1e-8
    kl1 = p1 * (torch.log(p1 + eps) - torch.log(p2 + eps))
    kl2 = p2 * (torch.log(p2 + eps) - torch.log(p1 + eps))
    return ((kl1 + kl2) * 0.5).mean()


def sensitivity_at_specificity_v2(y_true, y_prob, target_spec=0.90):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    idx = np.argmin(np.abs(spec - target_spec))
    return float(tpr[idx])


def compute_metrics_v2(y_true, y_prob):
    return {
        'auc'          : roc_auc_score(y_true, y_prob),
        'auprc'        : average_precision_score(y_true, y_prob),
        'brier'        : brier_score_loss(y_true, y_prob),
        'sens_at_90spec': sensitivity_at_specificity_v2(y_true, y_prob)
    }


def train_one_epoch_v2(model, loader, optimizer, scheduler, focal_loss, config, scaler):
    model.train()
    total_loss = 0
    for x_temp, x_stat, y in loader:
        x_temp, x_stat, y = x_temp.to(DEVICE), x_stat.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits1, sp_loss1 = model(x_temp, x_stat)
            logits2, sp_loss2 = model(x_temp, x_stat)
            task_loss = (focal_loss(logits1, y) + focal_loss(logits2, y)) * 0.5
            rdrop_loss = rdrop_kl_loss(logits1, logits2)
            sp_loss = (sp_loss1 + sp_loss2) * 0.5
            loss = task_loss + config['rdrop_alpha'] * rdrop_loss + sp_loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x_temp, x_stat, y in loader:
        x_temp, x_stat = x_temp.to(DEVICE), x_stat.to(DEVICE)
        logits, _ = model(x_temp, x_stat)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)


class SnapshotEnsemble:
    def __init__(self, k=5):
        self.k = k
        self.snapshots = []

    def update(self, val_auc, model):
        state = {k: v.clone() for k, v in model.state_dict().items()}
        self.snapshots.append((val_auc, state))
        self.snapshots.sort(key=lambda x: x[0], reverse=True)
        self.snapshots = self.snapshots[:self.k]

    @torch.no_grad()
    def predict(self, model, loader):
        all_probs_ensemble = []
        all_labels = None
        for auc, state in self.snapshots:
            model.load_state_dict(state)
            model.eval()
            labels, probs = evaluate(model, loader)
            all_probs_ensemble.append(probs)
            if all_labels is None:
                all_labels = labels
        ensemble_probs = np.mean(all_probs_ensemble, axis=0)
        return all_labels, ensemble_probs, all_probs_ensemble


def build_model_v2(config):
    return DSRNNv2(
        n_input=N_INPUT, n_static=N_STATIC, hidden_size=config['hidden_size'],
        n_lstm_layers=config['n_lstm_layers'], n_heads=config['n_heads'],
        attention_dim=config['attention_dim'], salience_hidden=config['salience_hidden'],
        dropout=config['dropout'], stoch_depth_p=config['stoch_depth_p'],
        lambda_sparse=config['lambda_sparse']
    )


def train_dsrnn_v2(config, run_idx=0, verbose=True):
    torch.manual_seed(SEED + run_idx)
    np.random.seed(SEED + run_idx)

    model = build_model_v2(config).to(DEVICE)
    if verbose:
        print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

    optimizer = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'], betas=(0.9, 0.999))
    total_steps = config['max_epochs'] * len(loaders['train'])
    scheduler = OneCycleLR(optimizer, max_lr=config['lr'], total_steps=total_steps,
                            pct_start=config['warmup_pct'], anneal_strategy='cos')
    focal_loss = FocalLossWithSmoothing(alpha=config['focal_alpha'], gamma=config['focal_gamma'],
                                        smoothing=config['label_smoothing'])
    scaler = GradScaler()
    ensemble = SnapshotEnsemble(k=config['n_snapshots'])

    best_val_auc, patience_count = 0, 0
    train_history = {'loss': [], 'val_auc': [], 'val_auprc': []}

    for epoch in range(1, config['max_epochs'] + 1):
        tr_loss = train_one_epoch_v2(model, loaders['train'], optimizer, scheduler, focal_loss, config, scaler)
        val_labels, val_probs = evaluate(model, loaders['val'])
        val_auc   = roc_auc_score(val_labels, val_probs)
        val_auprc = average_precision_score(val_labels, val_probs)

        train_history['loss'].append(tr_loss)
        train_history['val_auc'].append(val_auc)
        train_history['val_auprc'].append(val_auprc)

        ensemble.update(val_auc, model)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_count = 0
        else:
            patience_count += 1

        if verbose and epoch % 10 == 0:
            print(f'    Epoch {epoch:3d} | loss={tr_loss:.4f} | val_AUC={val_auc:.4f} | '
                  f'val_AUPRC={val_auprc:.4f} | best={best_val_auc:.4f}')
        if patience_count >= config['patience']:
            if verbose: print(f'    \u23f9 Early stopping di epoch {epoch}')
            break

    best_state = ensemble.snapshots[0][1]
    model.load_state_dict(best_state)
    test_labels, test_probs_single = evaluate(model, loaders['test'])
    metrics_single = compute_metrics_v2(test_labels, test_probs_single)

    _, test_probs_ensemble, _ = ensemble.predict(model, loaders['test'])
    metrics_ensemble = compute_metrics_v2(test_labels, test_probs_ensemble)

    return {
        'single': metrics_single, 'ensemble': metrics_ensemble, 'history': train_history,
        'model': model, 'ens_obj': ensemble, 'test_labels': test_labels,
        'test_probs_single': test_probs_single, 'test_probs_ensemble': test_probs_ensemble,
    }

print('\u2705 Training pipeline v2 siap')

## Bagian 17: Train COST (Improved) — 3 runs

In [ ]:
print(' Training DS-RNN v2 (COST Improved)...')
print('='*60)

all_runs_v2 = []
for run in range(CONFIG_V2['n_runs']):
    print(f"\n  Run {run+1}/{CONFIG_V2['n_runs']}:")
    result = train_dsrnn_v2(CONFIG_V2, run_idx=run, verbose=True)
    all_runs_v2.append(result)
    print(f'  \u2192 Single  AUC: {result["single"]["auc"]:.4f} | AUPRC: {result["single"]["auprc"]:.4f}')
    print(f'  \u2192 Ensemble AUC: {result["ensemble"]["auc"]:.4f} | AUPRC: {result["ensemble"]["auprc"]:.4f}')

print('\n' + '='*60)
print(' RINGKASAN DS-RNN v2 (3 runs):')
for key in ['single', 'ensemble']:
    aucs   = [r[key]['auc']   for r in all_runs_v2]
    auprcs = [r[key]['auprc'] for r in all_runs_v2]
    print(f'  {key.capitalize():10s}: AUC={np.mean(aucs):.4f}\u00b1{np.std(aucs):.4f} | '
          f'AUPRC={np.mean(auprcs):.4f}\u00b1{np.std(auprcs):.4f}')

best_run_v2 = max(all_runs_v2, key=lambda r: r['ensemble']['auc'])

## Bagian 18: Save COST (Improved) Predictions

In [ ]:
np.savez(
    PRED_DIR / 'cost_improved_predictions.npz',
    test_labels     = best_run_v2['test_labels'],
    probs_single    = best_run_v2['test_probs_single'],
    probs_ensemble  = best_run_v2['test_probs_ensemble'],
)
print(f'\u2705 Saved: {PRED_DIR / "cost_improved_predictions.npz"}')
print(f'   Single   AUC: {best_run_v2["single"]["auc"]:.4f}')
print(f'   Ensemble AUC: {best_run_v2["ensemble"]["auc"]:.4f}')

# Also persist model weights (useful if you need to re-inspect later)
model_path = SAVE_DIR / 'dsrnn_v2_best.pt'
torch.save({
    'model_state_dict' : best_run_v2['ens_obj'].snapshots[0][1],
    'config'            : CONFIG_V2,
    'val_auc'           : best_run_v2['ens_obj'].snapshots[0][0],
    'test_auc_single'   : best_run_v2['single']['auc'],
    'test_auc_ensemble' : best_run_v2['ensemble']['auc'],
    'norm_stats'        : norm_stats,
}, model_path)
print(f'\u2705 Model checkpoint saved: {model_path}')

## Bagian 19: Ablation Study — COST (Improved) Components (Table XXV)

In [ ]:
class DSRNNv2_Ablation(nn.Module):
    def __init__(self, n_input, n_static, hidden_size, n_lstm_layers,
                 n_heads, attention_dim, salience_hidden, dropout=0.3,
                 use_missingness_feat=True, use_bidirectional=True,
                 use_multihead_attn=True, use_positional_enc=True,
                 lambda_sparse=5e-4):
        super().__init__()
        self.hidden_size   = hidden_size
        self.use_miss_feat = use_missingness_feat
        self.use_bidir      = use_bidirectional
        self.use_mha         = use_multihead_attn
        self.use_pe           = use_positional_enc
        self.lambda_sparse = lambda_sparse

        actual_input = n_input if use_missingness_feat else (n_input // 3)

        self.input_proj = nn.Sequential(
            nn.Linear(actual_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos_enc = SinusoidalPE(hidden_size) if use_positional_enc else nn.Identity()
        self.salience = ClinicalSalienceV2(hidden_size + hidden_size, salience_hidden, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers,
                             batch_first=True, bidirectional=use_bidirectional,
                             dropout=dropout if n_lstm_layers > 1 else 0)
        bidir_out = hidden_size * (2 if use_bidirectional else 1)
        self.bidir_proj = nn.Sequential(
            nn.Linear(bidir_out, hidden_size), nn.LayerNorm(hidden_size), nn.GELU()
        )
        if use_multihead_attn:
            self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
        self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(attention_dim), nn.Dropout(dropout),
            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
            nn.Linear(attention_dim // 2, 1)
        )

    def forward(self, x_temporal, x_static, return_states=False):
        batch, seq, _ = x_temporal.shape
        if not self.use_miss_feat:
            x_temporal = x_temporal[:, :, :N_FEATURES]

        x_proj = self.input_proj(x_temporal)
        if self.use_pe:
            x_proj = self.pos_enc(x_proj)

        sparsity_loss = torch.tensor(0.0, device=x_temporal.device)
        h_prev = torch.zeros(batch, self.hidden_size, device=x_temporal.device)
        gated_seq = []
        for t in range(seq):
            x_t = x_proj[:, t, :]
            m_t, _ = self.salience(x_t, h_prev)
            x_gated = m_t * x_t + (1 - m_t) * x_t * 0.1
            gated_seq.append(x_gated.unsqueeze(1))
            h_prev = h_prev * 0.9 + x_gated.detach() * 0.1
            sparsity_loss = sparsity_loss + m_t.norm(p=1)

        x_gated_seq = torch.cat(gated_seq, dim=1)
        sparsity_loss = self.lambda_sparse * sparsity_loss / (seq * batch)

        lstm_out, _ = self.lstm(x_gated_seq)
        lstm_out = self.bidir_proj(lstm_out)

        if self.use_mha:
            lstm_out, _ = self.temporal_attn(lstm_out)

        z, _ = self.fusion(x_static, lstm_out)
        logits = self.classifier(z).squeeze(-1)
        if return_states: return logits, sparsity_loss, lstm_out, None
        return logits, sparsity_loss


ABLATION_VARIANTS = [
    ('Full v2 (semua improvement)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Missingness Features (#2)',
     dict(use_missingness_feat=False, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Bidirectional LSTM (#1)',
     dict(use_missingness_feat=True, use_bidirectional=False,
          use_multihead_attn=True, use_positional_enc=True)),
    ('w/o Multi-Head Attention (#1)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=False, use_positional_enc=True)),
    ('w/o Positional Encoding (#1)',
     dict(use_missingness_feat=True, use_bidirectional=True,
          use_multihead_attn=True, use_positional_enc=False)),
]


def run_ablation_variant(flags, config, n_runs=2):
    run_aucs, run_auprcs = [], []
    for run in range(n_runs):
        torch.manual_seed(SEED + run)
        model = DSRNNv2_Ablation(
            n_input=N_INPUT, n_static=N_STATIC, hidden_size=config['hidden_size'],
            n_lstm_layers=config['n_lstm_layers'], n_heads=config['n_heads'],
            attention_dim=config['attention_dim'], salience_hidden=config['salience_hidden'],
            dropout=config['dropout'], lambda_sparse=config['lambda_sparse'], **flags
        ).to(DEVICE)

        optimizer  = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        total_steps = 50 * len(loaders['train'])
        scheduler  = OneCycleLR(optimizer, max_lr=config['lr'], total_steps=total_steps,
                                pct_start=0.1, anneal_strategy='cos')
        focal_loss = FocalLossWithSmoothing(config['focal_alpha'], config['focal_gamma'],
                                            config['label_smoothing'])
        scaler = GradScaler()
        best_val_auc, best_state, patience = 0, None, 0

        for epoch in range(1, 51):
            train_one_epoch_v2(model, loaders['train'], optimizer, scheduler, focal_loss, config, scaler)
            vl, vp = evaluate(model, loaders['val'])
            va = roc_auc_score(vl, vp)
            if va > best_val_auc:
                best_val_auc = va
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
            if patience >= config['patience']: break

        model.load_state_dict(best_state)
        tl, tp = evaluate(model, loaders['test'])
        run_aucs.append(roc_auc_score(tl, tp))
        run_auprcs.append(average_precision_score(tl, tp))

    return np.mean(run_aucs), np.std(run_aucs), np.mean(run_auprcs), np.std(run_auprcs)


print(' Running Ablation Study (COST Improved components)...')
ablation_results_v2 = {}
for variant_name, flags in ABLATION_VARIANTS:
    print(f'  \u2192 {variant_name}...')
    m_auc, s_auc, m_auprc, s_auprc = run_ablation_variant(flags, CONFIG_V2, n_runs=2)
    ablation_results_v2[variant_name] = {'auc': m_auc, 'auc_std': s_auc, 'auprc': m_auprc, 'auprc_std': s_auprc}
    print(f'     AUC = {m_auc:.4f} \u00b1 {s_auc:.4f}')

print('\n ABLATION STUDY v2 (Table XXV):')
print('-'*75)
print(f'{"Variant":<40} {"AUC-ROC":>12} {"AUPRC":>8}')
print('-'*75)
for variant_name in ablation_results_v2:
    r = ablation_results_v2[variant_name]
    marker = ' \u2605' if 'Full' in variant_name else ''
    print(f'{variant_name:<40} {r["auc"]:>6.4f}\u00b1{r["auc_std"]:.4f} {r["auprc"]:>8.4f}{marker}')

## Bagian 20: Ablation — Optimization Strategy (optimization component comparison)

In [ ]:
# Architecture held fixed at Full v2; only loss/optimizer reverted to
# COST v1's original setup (Weighted BCE + Adam + CosineAnnealingLR,
# no Focal loss, no R-Drop, no OneCycleLR) - isolates the one bundled
# change from Table XXIX never previously ablated.

def run_optim_ablation(config, n_runs=2):
    run_aucs, run_auprcs = [], []
    for run in range(n_runs):
        torch.manual_seed(SEED + run)
        model = DSRNNv2_Ablation(
            n_input=N_INPUT, n_static=N_STATIC,
            hidden_size=config['hidden_size'], n_lstm_layers=config['n_lstm_layers'],
            n_heads=config['n_heads'], attention_dim=config['attention_dim'],
            salience_hidden=config['salience_hidden'], dropout=config['dropout'],
            lambda_sparse=config['lambda_sparse'],
            use_missingness_feat=True, use_bidirectional=True,
            use_multihead_attn=True, use_positional_enc=True,
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
        scheduler = CosineAnnealingLR(optimizer, T_max=50)
        pos_weight = torch.tensor([(1 - 0.138) / 0.138]).to(DEVICE)
        bce_loss = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        scaler = GradScaler()
        best_val_auc, best_state, patience = 0, None, 0

        for epoch in range(1, 51):
            model.train()
            for x_temp, x_stat, y in loaders['train']:
                x_temp, x_stat, y = x_temp.to(DEVICE), x_stat.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    logits, sparsity_loss = model(x_temp, x_stat)
                    loss = bce_loss(logits, y) + sparsity_loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            scheduler.step()

            vl, vp = evaluate(model, loaders['val'])
            va = roc_auc_score(vl, vp)
            if va > best_val_auc:
                best_val_auc = va
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
            if patience >= config['patience']: break

        model.load_state_dict(best_state)
        tl, tp = evaluate(model, loaders['test'])
        run_aucs.append(roc_auc_score(tl, tp))
        run_auprcs.append(average_precision_score(tl, tp))

    return np.mean(run_aucs), np.std(run_aucs), np.mean(run_auprcs), np.std(run_auprcs)


print(' Running Optimization-Strategy Ablation...')
optim_m_auc, optim_s_auc, optim_m_auprc, optim_s_auprc = run_optim_ablation(CONFIG_V2, n_runs=2)

# Fold into the same results dict as Table XXV, so it prints as one unified table
ablation_results_v2['w/o Optimization Strategy (BCE+Adam+CosineAnnealingLR)'] = {
    'auc': optim_m_auc, 'auc_std': optim_s_auc, 'auprc': optim_m_auprc, 'auprc_std': optim_s_auprc
}

print(f'\nw/o Optimization Strategy: AUC = {optim_m_auc:.4f} \u00b1 {optim_s_auc:.4f}, AUPRC = {optim_m_auprc:.4f}')
full_v2 = ablation_results_v2['Full v2 (semua improvement)']
print(f'   (compare to Full v2: AUC = {full_v2["auc"]:.4f} \u00b1 {full_v2["auc_std"]:.4f})')

print('\n COMPLETE TABLE XXV (all 6 variants, including Optimization Strategy):')
print('-'*80)
print(f'{"Variant":<55} {"AUC-ROC":>12} {"AUPRC":>8}')
print('-'*80)
for variant_name, r in ablation_results_v2.items():
    marker = ' \u2605' if 'Full' in variant_name else ''
    print(f'{variant_name:<55} {r["auc"]:>6.4f}\u00b1{r["auc_std"]:.4f} {r["auprc"]:>8.4f}{marker}')